# Bank Customer Churn — Prediction & Decision System

**Dataset:** IBM "Churn Modelling" sample — 10,000 zboun min bank fransa (البنك).
**Target:** `Exited` (1 = churned, 0 = stayed) — Binary Classification.

**Pipeline:**
1. EDA (Exploratory Data Analysis)
2. Feature Engineering
3. Preprocessing + handling class imbalance
4. Model training & comparison (Logistic Regression, Random Forest, XGBoost) with `RandomizedSearchCV`
5. Explainability with SHAP
6. Per-customer decision system (Risk Tier + Reasons + Recommended Action)
7. Export final report (Excel + CSV)


In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


## 1. Load Data & EDA

In [5]:
df_raw = pd.read_csv("/Users/sajaa7/Downloads/Churn_Modelling.csv.xls")

print(df_raw.shape)
print(df_raw.dtypes)
print(df_raw.isnull().sum())


(10000, 14)
RowNumber            int64
CustomerId           int64
Surname                str
CreditScore          int64
Geography              str
Gender                 str
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object
RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


In [6]:
print("Churn rate (Exited):")
print(df_raw['Exited'].value_counts(normalize=True))

df_raw.describe()


Churn rate (Exited):
Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64


,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [7]:
# Quick look at churn rate by key categorical variables
print("Churn rate by Geography:")
print(df_raw.groupby('Geography')['Exited'].mean().round(3))

print("\nChurn rate by Gender:")
print(df_raw.groupby('Gender')['Exited'].mean().round(3))

print("\nChurn rate by NumOfProducts:")
print(df_raw.groupby('NumOfProducts')['Exited'].mean().round(3))

print("\nChurn rate by IsActiveMember:")
print(df_raw.groupby('IsActiveMember')['Exited'].mean().round(3))


Churn rate by Geography:
Geography
France     0.162
Germany    0.324
Spain      0.167
Name: Exited, dtype: float64

Churn rate by Gender:
Gender
Female    0.251
Male      0.165
Name: Exited, dtype: float64

Churn rate by NumOfProducts:
NumOfProducts
1    0.277
2    0.076
3    0.827
4    1.000
Name: Exited, dtype: float64

Churn rate by IsActiveMember:
IsActiveMember
0    0.269
1    0.143
Name: Exited, dtype: float64


## 2. Feature Engineering

Feature يلي زادت إشارة الـchurn:
- `HighProductCount`: 3+ منتجات → إشارة قوية جدًا (82-100% churn)
- `InactiveSenior`: زبون غير نشط وعمره 50+
- `IsGermany`, `BalanceSalaryRatio`, `TenureByAge`, `CreditScoreBin`, `AgeGroup`, `ProductsPerTenure`


In [8]:
df = df_raw.drop(columns=['RowNumber', 'Surname'])  # CustomerId يضل للتقرير النهائي، بينشال بس عند التدريب

# ---- Feature engineering ----
df['ZeroBalance'] = (df['Balance'] == 0).astype(int)
df['BalanceSalaryRatio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
df['HighProductCount'] = (df['NumOfProducts'] >= 3).astype(int)
df['IsGermany'] = (df['Geography'] == 'Germany').astype(int)
df['TenureByAge'] = df['Tenure'] / df['Age']
df['CreditScoreBin'] = pd.cut(df['CreditScore'], bins=[0, 580, 670, 740, 800, 850],
                               labels=['Poor', 'Fair', 'Good', 'VeryGood', 'Excellent'])
df['AgeGroup'] = pd.cut(df['Age'], bins=[17, 30, 40, 50, 60, 100],
                         labels=['18-30', '31-40', '41-50', '51-60', '60+'])
df['InactiveSenior'] = ((df['IsActiveMember'] == 0) & (df['Age'] >= 50)).astype(int)
df['ProductsPerTenure'] = df['NumOfProducts'] / (df['Tenure'] + 1)

print(df.shape)
print(df.columns.tolist())
df[['ZeroBalance', 'BalanceSalaryRatio', 'HighProductCount', 'IsGermany',
    'TenureByAge', 'InactiveSenior']].describe()


(10000, 21)
['CustomerId', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'ZeroBalance', 'BalanceSalaryRatio', 'HighProductCount', 'IsGermany', 'TenureByAge', 'CreditScoreBin', 'AgeGroup', 'InactiveSenior', 'ProductsPerTenure']


,ZeroBalance,BalanceSalaryRatio,HighProductCount,IsGermany,TenureByAge,InactiveSenior
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,0.361700,3.790150,0.032600,0.250900,0.137936,0.048400
std,0.480517,100.055758,0.177596,0.433553,0.089506,0.214621
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.064516,0.000000
50%,0.000000,0.746998,0.000000,0.000000,0.129032,0.000000
75%,1.000000,1.514002,0.000000,1.000000,0.200000,0.000000
max,1.000000,9770.883148,1.000000,1.000000,0.555556,1.000000


In [9]:
df.to_csv('features.csv', index=False)
print("Saved features.csv")


Saved features.csv


## 3. Train/Test Split & Preprocessing Pipeline

In [10]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
import joblib
import os

os.makedirs('outputs', exist_ok=True)

target = 'Exited'
id_col = 'CustomerId'
X = df.drop(columns=[id_col, target])
y = df[target]

cat_cols = ['Geography', 'Gender', 'CreditScoreBin', 'AgeGroup']
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train churn rate:", round(y_train.mean(), 4), "Test churn rate:", round(y_test.mean(), 4))

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), cat_cols)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


Train shape: (8000, 19) Test shape: (2000, 19)
Train churn rate: 0.2037 Test churn rate: 0.2035


## 4. Model Training & Comparison

منقارن 3 موديلات، كل واحد بمعالجة إمبالانس مختلفة:
- **Logistic Regression**: `class_weight='balanced'`
- **Random Forest**: `class_weight='balanced'` + `RandomizedSearchCV`
- **XGBoost**: `scale_pos_weight` + `RandomizedSearchCV`
- **XGBoost + SMOTE**: كمقارنة إضافية لطريقة oversampling


In [11]:
# ---------------- Logistic Regression (baseline) ----------------
logreg_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
logreg_pipe.fit(X_train, y_train)
logreg_proba = logreg_pipe.predict_proba(X_test)[:, 1]

print("=== Logistic Regression (baseline, class_weight=balanced) ===")
print("ROC-AUC:", round(roc_auc_score(y_test, logreg_proba), 4))
print("PR-AUC :", round(average_precision_score(y_test, logreg_proba), 4))


=== Logistic Regression (baseline, class_weight=balanced) ===
ROC-AUC: 0.8502
PR-AUC : 0.6887


In [12]:
# ---------------- Random Forest (tuned) ----------------
rf_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])
rf_params = {
    'clf__n_estimators': [200, 400, 600],
    'clf__max_depth': [4, 6, 8, 12, None],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2']
}
rf_search = RandomizedSearchCV(rf_pipe, rf_params, n_iter=25, scoring='roc_auc',
                                cv=cv, random_state=42, n_jobs=-1)
rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_
rf_proba = rf_best.predict_proba(X_test)[:, 1]

print("=== Random Forest (tuned, class_weight=balanced) ===")
print("Best params:", rf_search.best_params_)
print("ROC-AUC:", round(roc_auc_score(y_test, rf_proba), 4))
print("PR-AUC :", round(average_precision_score(y_test, rf_proba), 4))


=== Random Forest (tuned, class_weight=balanced) ===
Best params: {'clf__n_estimators': 600, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 4, 'clf__max_features': 'sqrt', 'clf__max_depth': 12}
ROC-AUC: 0.8612
PR-AUC : 0.7063


In [13]:
# ---------------- XGBoost (tuned, scale_pos_weight) ----------------
neg, pos = np.bincount(y_train)
spw = neg / pos

xgb_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', xgb.XGBClassifier(objective='binary:logistic', eval_metric='auc',
                               scale_pos_weight=spw, random_state=42, n_jobs=-1,
                               tree_method='hist'))
])
xgb_params = {
    'clf__n_estimators': [200, 300, 500],
    'clf__max_depth': [3, 4, 5, 6],
    'clf__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'clf__subsample': [0.7, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0],
    'clf__min_child_weight': [1, 3, 5]
}
xgb_search = RandomizedSearchCV(xgb_pipe, xgb_params, n_iter=30, scoring='roc_auc',
                                 cv=cv, random_state=42, n_jobs=-1)
xgb_search.fit(X_train, y_train)
xgb_best = xgb_search.best_estimator_
xgb_proba = xgb_best.predict_proba(X_test)[:, 1]

print("=== XGBoost (tuned, scale_pos_weight) ===")
print("Best params:", xgb_search.best_params_)
print("ROC-AUC:", round(roc_auc_score(y_test, xgb_proba), 4))
print("PR-AUC :", round(average_precision_score(y_test, xgb_proba), 4))


=== XGBoost (tuned, scale_pos_weight) ===
Best params: {'clf__subsample': 1.0, 'clf__n_estimators': 200, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__colsample_bytree': 0.6}
ROC-AUC: 0.8699
PR-AUC : 0.7135


In [14]:
# ---------------- XGBoost + SMOTE (alternative imbalance strategy) ----------------
xgb_smote_pipe = ImbPipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('clf', xgb.XGBClassifier(objective='binary:logistic', eval_metric='auc',
                               random_state=42, n_jobs=-1,
                               n_estimators=300, max_depth=4, learning_rate=0.05))
])
xgb_smote_pipe.fit(X_train, y_train)
smote_proba = xgb_smote_pipe.predict_proba(X_test)[:, 1]

print("=== XGBoost + SMOTE ===")
print("ROC-AUC:", round(roc_auc_score(y_test, smote_proba), 4))
print("PR-AUC :", round(average_precision_score(y_test, smote_proba), 4))


=== XGBoost + SMOTE ===
ROC-AUC: 0.8648
PR-AUC : 0.7153


In [15]:
# ---------------- Model comparison summary ----------------
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest (tuned)', 'XGBoost (tuned)', 'XGBoost + SMOTE'],
    'ROC-AUC': [
        roc_auc_score(y_test, logreg_proba),
        roc_auc_score(y_test, rf_proba),
        roc_auc_score(y_test, xgb_proba),
        roc_auc_score(y_test, smote_proba),
    ],
    'PR-AUC': [
        average_precision_score(y_test, logreg_proba),
        average_precision_score(y_test, rf_proba),
        average_precision_score(y_test, xgb_proba),
        average_precision_score(y_test, smote_proba),
    ]
}).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print(comparison.round(4))

# XGBoost (tuned) هو الأفضل -> بنعتمده كموديل الإنتاج
best_model = xgb_best
joblib.dump(best_model, 'outputs/churn_model.joblib')
joblib.dump({'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test,
             'cat_cols': cat_cols, 'num_cols': num_cols},
            'outputs/data_splits.joblib')
print("\nSaved best model -> outputs/churn_model.joblib")


                   Model  ROC-AUC  PR-AUC
0        XGBoost (tuned)   0.8699  0.7135
1        XGBoost + SMOTE   0.8648  0.7153
2  Random Forest (tuned)   0.8612  0.7063
3    Logistic Regression   0.8502  0.6887

Saved best model -> outputs/churn_model.joblib


## 5. Explainability (SHAP)

In [16]:
import shap

prep = best_model.named_steps['prep']
clf = best_model.named_steps['clf']

X_test_t = prep.transform(X_test)
feature_names = prep.get_feature_names_out()
X_test_df = pd.DataFrame(X_test_t, columns=feature_names, index=X_test.index)

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_df)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False)

print("=== Top 15 global feature importance (mean |SHAP|) ===")
print(importance.head(15))


ModuleNotFoundError: No module named 'shap'

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_test_df, show=False, max_display=15)
plt.tight_layout()
plt.savefig('outputs/shap_summary.png', dpi=130)
plt.close()

joblib.dump({'shap_values': shap_values, 'feature_names': feature_names,
             'X_test_df': X_test_df, 'explainer_expected_value': explainer.expected_value},
            'outputs/shap_data.joblib')
print("Saved outputs/shap_summary.png and outputs/shap_data.joblib")


## 6. Per-Customer Decision System

لكل زبون بالـtest set:
- **ChurnProbability**: من `predict_proba`
- **RiskTier**: Critical / High / Medium / Low (حسب عتبات الاحتمال)
- **TopReason 1-3**: أعلى 3 فيتشرز أثرت عالقرار (من SHAP، حسب الإشارة الموجبة يلي بتزود احتمال الـchurn)
- **RecommendedAction**: إجراء مختلف حسب السبب الرئيسي (مش نفس الحل للكل)


In [ ]:
churn_proba = best_model.predict_proba(X_test)[:, 1]

def risk_tier(p):
    if p >= 0.70:
        return 'Critical'
    elif p >= 0.45:
        return 'High'
    elif p >= 0.20:
        return 'Medium'
    else:
        return 'Low'

def top_reasons(row_shap, feat_names, n=3):
    # أعلى n فيتشرز حسب |SHAP|، بس يلي بيزيدوا احتمال الترك (SHAP > 0)
    idx_sorted = np.argsort(-row_shap)
    reasons = []
    for i in idx_sorted:
        if row_shap[i] > 0:
            reasons.append(feat_names[i])
        if len(reasons) == n:
            break
    while len(reasons) < n:
        reasons.append(None)
    return reasons

def recommended_action(reasons, num_products):
    top = reasons[0] if reasons[0] else ''
    if num_products >= 3:
        return 'مراجعة فورية من مدير علاقة (Relationship Manager)'
    if 'IsActiveMember' in top or 'InactiveSenior' in top:
        return 'حملة إعادة تفعيل (Re-engagement campaign)'
    if 'Age' in top:
        return 'عرض مخصص حسب الفئة العمرية + مكالمة استباقية'
    if 'Germany' in top:
        return 'مراجعة تسعير/خدمة السوق الألماني'
    return 'متابعة دورية + عرض احتفاظ عام (retention offer)'

report_rows = []
customer_ids = df.loc[X_test.index, 'CustomerId'].values
num_products_arr = X_test['NumOfProducts'].values

for i, cust_id in enumerate(customer_ids):
    reasons = top_reasons(shap_values[i], feature_names, n=3)
    prob = churn_proba[i]
    report_rows.append({
        'CustomerId': cust_id,
        'ChurnProbability': round(float(prob), 4),
        'RiskTier': risk_tier(prob),
        'ActualExited': int(y_test.iloc[i]),
        'TopReason1': reasons[0],
        'TopReason2': reasons[1],
        'TopReason3': reasons[2],
        'RecommendedAction': recommended_action(reasons, num_products_arr[i])
    })

customer_risk_report = pd.DataFrame(report_rows)
customer_risk_report.to_csv('outputs/customer_risk_report.csv', index=False)
print(customer_risk_report.shape)
customer_risk_report.head(10)


In [ ]:
# ---- Validation: هل الـRiskTier فعلاً بيفرز الزباين؟ ----
tier_validation = customer_risk_report.groupby('RiskTier')['ActualExited'].agg(['mean', 'count']).round(3)
tier_validation = tier_validation.reindex(['Critical', 'High', 'Medium', 'Low'])
print("Actual churn rate per RiskTier:")
print(tier_validation)


In [ ]:
print("Distribution of RecommendedAction across customers:")
print(customer_risk_report['RecommendedAction'].value_counts())


## 7. Export Final Report (Excel + CSV)

In [ ]:
with pd.ExcelWriter('outputs/churn_retention_report.xlsx', engine='xlsxwriter') as writer:
    # Executive summary
    summary = pd.DataFrame({
        'Metric': ['Total customers (test set)', 'Overall churn rate (test)',
                   'Best model', 'Best ROC-AUC', 'Critical tier actual churn %',
                   'Low tier actual churn %'],
        'Value': [len(customer_risk_report), f"{y_test.mean():.1%}",
                  'XGBoost (tuned)', f"{roc_auc_score(y_test, churn_proba):.3f}",
                  f"{tier_validation.loc['Critical', 'mean']:.1%}",
                  f"{tier_validation.loc['Low', 'mean']:.1%}"]
    })
    summary.to_excel(writer, sheet_name='Executive Summary', index=False)

    # Action plan (aggregated by RecommendedAction / RiskTier)
    action_plan = customer_risk_report.groupby(['RiskTier', 'RecommendedAction']).size() \
        .reset_index(name='CustomerCount').sort_values(['RiskTier', 'CustomerCount'], ascending=[True, False])
    action_plan.to_excel(writer, sheet_name='Action Plan', index=False)

    # Full per-customer report
    customer_risk_report.to_excel(writer, sheet_name='Customer Risk Report', index=False)

print("Saved outputs/churn_retention_report.xlsx")
